<a href="https://colab.research.google.com/github/davidfwallace/fraude-abertura-conta/blob/main/notebooks/01_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/fraude-abertura-conta/data/Base.csv')

pd.set_option('display.max_columns', None)
df.describe().T

Mounted at /content/drive


,count,mean,std,min,25%,50%,75%,max
fraud_bool,1000000.0,0.011029,0.104438,0.000000e+00,0.000000,0.000000,0.000000,1.000000
income,1000000.0,0.562696,0.290343,1.000000e-01,0.300000,0.600000,0.800000,0.900000
name_email_similarity,1000000.0,0.493694,0.289125,1.434550e-06,0.225216,0.492153,0.755567,0.999999
prev_address_months_count,1000000.0,16.718568,44.046230,-1.000000e+00,-1.000000,-1.000000,12.000000,383.000000
current_address_months_count,1000000.0,86.587867,88.406599,-1.000000e+00,19.000000,52.000000,130.000000,428.000000
customer_age,1000000.0,33.689080,12.025799,1.000000e+01,20.000000,30.000000,40.000000,90.000000
days_since_request,1000000.0,1.025705,5.381835,4.036860e-09,0.007193,0.015176,0.026331,78.456904
intended_balcon_amount,1000000.0,8.661499,20.236155,-1.553055e+01,-1.181488,-0.830507,4.984176,112.956928
zip_count_4w,1000000.0,1572.692049,1005.374565,1.000000e+00,894.000000,1263.000000,1944.000000,6700.000000
velocity_6h,1000000.0,5665.296605,3009.380665,-1.706031e+02,3436.365848,5319.769349,7680.717827,16715.565404


In [ ]:
treino = df[df['month'] <= 5]

cols_ausentes = ['prev_address_months_count', 'current_address_months_count',
                 'bank_months_count', 'intended_balcon_amount',
                 'session_length_in_minutes', 'device_distinct_emails_8w']

for c in cols_ausentes:
    ausente = treino[c] < 0
    print(f"{c:32s} | % ausente: {ausente.mean():6.1%} | "
          f"fraude se ausente: {treino.loc[ausente, 'fraud_bool'].mean():.2%} | "
          f"se presente: {treino.loc[~ausente, 'fraud_bool'].mean():.2%}")

prev_address_months_count        | % ausente:  71.5% | fraude se ausente: 1.32% | se presente: 0.28%
current_address_months_count     | % ausente:   0.4% | fraude se ausente: 0.30% | se presente: 1.03%
bank_months_count                | % ausente:  25.1% | fraude se ausente: 1.54% | se presente: 0.85%
intended_balcon_amount           | % ausente:  73.7% | fraude se ausente: 1.22% | se presente: 0.47%
session_length_in_minutes        | % ausente:   0.2% | fraude se ausente: 0.86% | se presente: 1.03%
device_distinct_emails_8w        | % ausente:   0.0% | fraude se ausente: 0.97% | se presente: 1.03%


In [3]:
cats = ['payment_type', 'employment_status', 'housing_status', 'source', 'device_os']

for c in cats:
    resumo = treino.groupby(c)['fraud_bool'].agg(['count', 'mean'])
    resumo['mean'] = (resumo['mean'] * 100).round(2)
    resumo.columns = ['qtd', '% fraude']
    print(f"\n=== {c}")
    print(resumo.sort_values('% fraude', ascending=False))


=== payment_type
                 qtd  % fraude
payment_type                  
AC            198579      1.58
AB            288182      1.04
AD             99060      0.98
AA            208920      0.50
AE               248      0.40

=== employment_status
                      qtd  % fraude
employment_status                  
CC                  31247      2.34
CG                    372      1.34
CA                 571543      1.13
CB                 115450      0.64
CD                  22005      0.40
CE                  18143      0.21
CF                  36229      0.19

=== housing_status
                   qtd  % fraude
housing_status                  
BA              129775      3.52
BD               17956      0.87
BC              301879      0.59
BB              204590      0.58
BG                 222      0.45
BF                1358      0.44
BE              139209      0.32

=== source
             qtd  % fraude
source                    
TELEAPP     5737      1.39
INTERNET

In [4]:
por_mes = treino.groupby('month').agg(
    qtd=('fraud_bool', 'size'),
    pct_fraude=('fraud_bool', 'mean'),
    credit_risk_med=('credit_risk_score', 'median'),
    limite_med=('proposed_credit_limit', 'median'),
    velocity_4w_med=('velocity_4w', 'median'),
)
por_mes['pct_fraude'] = (por_mes['pct_fraude'] * 100).round(2)
print(por_mes)

          qtd  pct_fraude  credit_risk_med  limite_med  velocity_4w_med
month                                                                  
0      132440        1.13            106.0       200.0      6343.590545
1      127620        0.94             99.0       200.0      5446.323889
2      136979        0.87            106.0       200.0      5462.167155
3      150936        0.92            138.0       200.0      4909.927613
4      127691        1.14            130.0       200.0      4695.658606
5      119323        1.18            133.0       200.0      4267.070964


## Resumo da Análise Exploratória

> Toda a EDA foi feita **apenas nos meses de treino (0–5)**. Os meses 6–7 são reservados como conjunto de teste e não foram inspecionados, para não influenciar decisões de modelagem.

### 1. Visão geral
- 1.000.000 de pedidos de abertura de conta, 30 atributos + alvo (`fraud_bool`) + `month`.
- **Taxa de fraude: ~1,1%** (~1,02% no treino). Um modelo que classifica tudo como legítimo teria 98,9% de acurácia e seria inútil → **acurácia descartada como métrica**. Usaremos **recall a 5% de FPR** (métrica de negócio) e **ROC AUC**.
- Divisão temporal: treino (meses 0–5) ≈ 795 mil linhas; teste (meses 6–7) ≈ 205 mil linhas (~20%).

### 2. Valores ausentes são informativos
Valores negativos (-1, ou qualquer negativo em `intended_balcon_amount`) indicam **ausência**, conforme o dicionário de dados. A ausência está associada a mais fraude:

| Atributo | % ausente | Fraude se ausente | Fraude se presente | Razão |
|---|---|---|---|---|
| `prev_address_months_count` | 71,5% | 1,32% | 0,28% | **~4,7×** |
| `intended_balcon_amount` | 73,7% | 1,22% | 0,47% | ~2,6× |
| `bank_months_count` | 25,1% | 1,54% | 0,85% | ~1,8× |

- Em `current_address_months_count`, `session_length_in_minutes` e `device_distinct_emails_8w`, a ausência é rara (≤ 0,4%), com poucas fraudes envolvidas, então as diferenças não são conclusivas.
- `credit_risk_score` tem valores negativos **legítimos** (a escala do score inclui negativos), que não são ausência.
- **Decisão:** não imputar pela média, o que apagaria o sinal. Modelos de árvore recebem o -1 como valor próprio; modelos lineares recebem indicadores binários de ausência.

### 3. Atributos categóricos
Sinais fortes (categorias grandes, taxa bem acima da média de ~1,02%):
- `housing_status = BA`: 3,52% (~130 mil linhas, **~3,4×** a média)
- `employment_status = CC`: 2,34% (~31 mil linhas)
- `device_os = windows`: 2,30% (~213 mil linhas)

Categorias raras (`payment_type = AE`, `employment_status = CG`, `housing_status = BG/BF`) têm entre 1 e 6 fraudes, o que é pouco para estimar uma taxa confiável.
- **Decisão:** agrupar categorias raras em `"outros"` para modelos lineares; one-hot encoding (poucas categorias por coluna).

### 4. Colunas sem informação
- `device_fraud_count` é **constante (sempre 0)** → será removida.
- `customer_age` (décadas) e `income` (faixas de 0,1 a 0,9) já vêm discretizados por anonimização.

### 5. Mudança ao longo do tempo (drift)
| Mês | % fraude | `credit_risk_score` (mediana) | `velocity_4w` (mediana) |
|---|---|---|---|
| 0 | 1,13 | 106 | 6.344 |
| 1 | 0,94 | 99 | 5.446 |
| 2 | 0,87 | 106 | 5.462 |
| 3 | 0,92 | 138 | 4.910 |
| 4 | 1,14 | 130 | 4.696 |
| 5 | 1,18 | 133 | 4.267 |

- `velocity_4w` cai de forma consistente (~33% em 6 meses): **drift gradual**. Nos meses de teste, é provável que haja valores fora da faixa vista no treino.
- `credit_risk_score` muda de patamar entre os meses 2 e 3: **mudança abrupta de nível**.
- A taxa de fraude também varia entre meses (0,87% a 1,18%).
- **Decisão:** validação **temporal com janela crescente** (treina 0–2 → valida 3; 0–3 → 4; 0–4 → 5), em vez de validação cruzada aleatória, que daria estimativas otimistas ao "prever o passado". O modelo final é treinado nos meses 0–5 e avaliado **uma única vez** nos meses 6–7.